# PPO + LSTM Trading — Kaggle Training Notebook

Notebook này dùng để **train PPO + LSTM** cho dự án trading trên Kaggle.

| Thành phần | Vai trò |
|-----------|---------|
| `src/environment/trading_env.py` | Môi trường Gymnasium cho trading |
| `src/models/lstm.py` | Mạng `PPOLSTMActorCritic` (LSTM + Actor-Critic) |
| `src/agents/ppo_agent.py` | `PPOAgent` + `RolloutBuffer` (thu thập rollout, update PPO) |
| `src/training/PPO.py` | Training loop chính (`train_ppo`) |
| `Conf/ppo_conf.yaml` | Cấu hình hyperparameter (dễ chỉnh trên Kaggle) |

Kịch bản chạy:
1. Cài đặt thư viện cần thiết
2. Clone repo từ GitHub vào `/kaggle/working/repo`
3. Đọc config từ `Conf/ppo_conf.yaml`
4. Gọi `train_ppo(config)` để train và log kết quả bởi `TrainingLogger`.

In [ ]:
# 1. Cài đặt thư viện cần thiết (nếu thiếu)

import subprocess
import sys

# Cài gymnasium (thường chưa có sẵn trên Kaggle)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gymnasium"], check=True)

print("Thư viện đã sẵn sàng.")

## 2. Clone GitHub Repo vào `/kaggle/working/repo`

Sử dụng Kaggle Secret `GH_TOKEN` để clone repo chứa mã nguồn PPO + LSTM.

In [ ]:
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import subprocess
import sys

# =============================================================
# CẬP NHẬT TÊN REPO TẠI ĐÂY
# =============================================================
GITHUB_REPO  = "sin0235/Project" 

user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GH_TOKEN")
BRANCH       = "master"


import subprocess, sys
from pathlib import Path

CLONE_DIR   = Path("/kaggle/working/repo")
WORKING_DIR = Path("/kaggle/working")

if CLONE_DIR.exists():
    result = subprocess.run(
        ["git", "-C", str(CLONE_DIR), "pull"],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
else:
    repo_url = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    result = subprocess.run(
        ["git", "clone", "--depth=1", "-b", BRANCH, repo_url, str(CLONE_DIR)],
        text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )

print(result.stdout)
if result.returncode != 0:
    raise RuntimeError("Git clone/pull thất bại. Kiểm tra lại TOKEN và REPO.")

PROJECT_ROOT = CLONE_DIR
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

## 3. Đọc cấu hình PPO từ `Conf/ppo_conf.yaml`

File config nằm trong repo vừa clone, dùng để điều chỉnh nhanh hyperparameter khi train trên Kaggle.

In [ ]:
import yaml
import numpy as np
from pprint import pprint

CONF_PATH = PROJECT_ROOT / "Conf" / "ppo_conf.yaml"

with open(CONF_PATH, "r", encoding="utf-8") as f:
    ppo_cfg = yaml.safe_load(f)

# -------------------------------------------------------------
# OVERRIDE đường dẫn data khi chạy trên Kaggle
# -------------------------------------------------------------
# Dataset Kaggle: /kaggle/input/datasets/phctontrn/dataset-trading-rl
# Giả định các file ACB.csv, BID.csv,... nằm trực tiếp trong thư mục này.
ppo_cfg["data_path"] = "/kaggle/input/datasets/phctontrn/dataset-trading-rl"

print("Đường dẫn config:", CONF_PATH)
print("Data path (Kaggle):", ppo_cfg["data_path"])
print("\nCấu hình PPO (rút gọn):")
keys_to_show = [
    "tickers", "features", "window_size",
    "train_ratio", "val_ratio", "test_ratio",
    "initial_balance", "fee_rate", "reward_name", "reward_window",
    "learning_rate", "n_steps", "batch_size", "total_timesteps",
]
summary = {k: ppo_cfg.get(k) for k in keys_to_show}
pprint(summary)

## 4. Chạy training PPO + LSTM

Cell dưới sẽ gọi trực tiếp `train_ppo(config)` trong `src/training/PPO.py`.

- Logger sẽ tự tạo `run_id` và ghi log vào `results/runs/<run_id>` bên trong repo
- Khi muốn thử hyperparameters khác, chỉ cần chỉnh `Conf/ppo_conf.yaml` rồi chạy lại cell này.

In [ ]:
import sys

# Đảm bảo PROJECT_ROOT đã nằm trong sys.path từ cell clone repo
sys.path.insert(0, str(PROJECT_ROOT))

from src.training.PPO import train_ppo

# Có thể override thêm một số hyperparameters ngay tại notebook nếu muốn, ví dụ:
# ppo_cfg["total_timesteps"] = 200_000
# ppo_cfg["learning_rate"] = 2e-4

agent, test_metrics = train_ppo(ppo_cfg)

print("\nKết quả test trung bình (tóm tắt):")
for k, v in sorted(test_metrics.items()):
    print(f"  {k}: {v}")

## 5. Eval chi tiết và xem lại kết quả

Cell dưới cho phép bạn **chạy lại eval** (không train thêm) từ checkpoint phù hợp nhất của run gần nhất: ưu tiên `best_model.pt`, nếu thiếu thì fallback sang `final_model.pt` hoặc `checkpoint_*.pt`, rồi in full metrics để tiện so sánh giữa các run.

In [ ]:
import sys
from pathlib import Path
import yaml
import numpy as np

# Đảm bảo PROJECT_ROOT đã được set ở cell clone repo
sys.path.insert(0, str(PROJECT_ROOT))

from src.training.PPO import resolve_eval_run, resolve_ppo_config
from src.utils.metrics import compute_all, format_report
from src.environment.trading_env import TradingEnv
from src.utils.data_splitter import load_data, split_by_ratio

# Eval lại: tự tìm run mới nhất còn dùng được.
# Ưu tiên config.json của run; nếu thiếu sẽ cố suy luận shape model từ checkpoint.

from src.models.lstm import PPOLSTMActorCritic
from src.agents.ppo_agent import PPOAgent

base_cfg = resolve_ppo_config()
base_cfg["data_path"] = "/kaggle/input/datasets/phctontrn/dataset-trading-rl"

results_root = PROJECT_ROOT / "results" / "runs"
if not results_root.exists():
    print("Chưa tìm thấy thư mục results/runs trong repo. Hãy chạy cell train trước.")
else:
    resolved = resolve_eval_run(
        results_root,
        base_config=base_cfg,
        overrides={
            "data_path": "/kaggle/input/datasets/phctontrn/dataset-trading-rl",
            "device": "auto",
        },
    )
    if resolved["run_dir"] is None:
        print("Không tìm thấy run nào đủ artifact để eval.")
        if resolved["skipped_runs"]:
            print("Các run bị bỏ qua:")
            for item in resolved["skipped_runs"]:
                print(" -", item["run_id"], "->", item["reason"])
    else:
        latest_run = resolved["run_dir"]
        ckpt_path = resolved["ckpt_path"]
        ckpt_source = resolved["ckpt_source"]
        cfg = resolved["config"]
        print("Latest evaluable run:", latest_run.name)
        print("Config source:", resolved["config_source"])
        print("Checkpoint source:", ckpt_source)
        if resolved["skipped_runs"]:
            print("Skipped runs before this selection:")
            for item in resolved["skipped_runs"]:
                print(" -", item["run_id"], "->", item["reason"])

        data_dict = load_data(tickers=cfg["tickers"], data_path=cfg["data_path"])
        split = split_by_ratio(
            data_dict,
            train_ratio=cfg["train_ratio"],
            val_ratio=cfg["val_ratio"],
            test_ratio=cfg["test_ratio"],
        )
        print("Data split summary:", split.summary())

        test_env = TradingEnv(
            tickers=cfg["tickers"],
            mode="continuous",
            initial_balance=cfg["initial_balance"],
            fee_rate=cfg["fee_rate"],
            window_size=cfg["window_size"],
            data_dict=split.test,
            features=cfg["features"],
            max_steps=cfg["max_steps_eval"],
            random_start=False,
            reward_scaling=cfg["reward_scaling"],
            reward_name=cfg["reward_name"],
            reward_kwargs={"window": cfg["reward_window"]},
            print_verbosity=999999,
        )

        state_space = test_env.state_space
        n_stocks = state_space.n_stocks
        n_features = state_space.n_features

        model = PPOLSTMActorCritic(
            n_stocks=n_stocks,
            n_features=n_features,
            seq_len=cfg["window_size"],
            hidden_size=cfg["hidden_size"],
            num_layers=cfg["num_layers"],
            dropout=cfg["dropout"],
            log_std_init=cfg["log_std_init"],
        )

        agent = PPOAgent(
            model=model,
            lr=cfg["learning_rate"],
            gamma=cfg["gamma"],
            gae_lambda=cfg["gae_lambda"],
            clip_range=cfg["clip_range"],
            ent_coef=cfg["ent_coef"],
            vf_coef=cfg["vf_coef"],
            max_grad_norm=cfg["max_grad_norm"],
            target_kl=cfg["target_kl"],
            n_epochs=cfg["n_epochs"],
            batch_size=cfg["batch_size"],
            device=cfg["device"],
        )

        print(f"Đang load checkpoint ({ckpt_source}):", ckpt_path)
        agent.load(str(ckpt_path))

        # Đánh giá trên test set
        print("\nĐang eval trên test set...")
        values_list = agent.evaluate(test_env, state_space, n_episodes=cfg["n_eval_episodes"])
        metrics_list = [compute_all(v, cfg["initial_balance"]) for v in values_list]

        avg_metrics = {}
        for key in metrics_list[0]:
            vals = [m[key] for m in metrics_list if key in m]
            avg_metrics[key] = float(np.mean(vals))

        print("\n=== TEST METRICS (TRUNG BÌNH) ===")
        print(format_report(avg_metrics))